In [24]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import graphviz # Para visualização da árvore
from sklearn.tree import export_graphviz # Função para exportar a árvore

In [25]:
# Definindo as opções para cada variável
opcoes_sim_nao = ['Sim', 'Nao']
opcoes_urgencia = ['Alta', 'Media', 'Baixa']
opcoes_clientes = [f'Cliente_{i}' for i in range(1, 6)] # 5 clientes de exemplo
opcoes_etiquetas = ['Etiqueta_105mm', 'Etiqueta_64mm'] # Apenas dois tipos de etiqueta para simplificar

# Número de exemplos (linhas) que queremos simular
num_amostras = 200

# Gerando dados simulados para as variáveis
data_simulada_simplificada = {
    'Estoque_OK': np.random.choice(opcoes_sim_nao, num_amostras, p=[0.7, 0.3]), # 70% chance de ter estoque OK
    'Dias_Desde_Pedido': np.random.randint(1, 30, num_amostras), # Dias desde que o pedido foi feito (1 a 29 dias)
    'Urgencia_Pedido': np.random.choice(opcoes_urgencia, num_amostras, p=[0.4, 0.3, 0.3]), # 40% Alta, 30% Media, 30% Baixa
    'Nome_Cliente': np.random.choice(opcoes_clientes, num_amostras),
    'Tempo_Medio_Producao_Min': np.random.randint(20, 180, num_amostras), # Tempo de produção em minutos (20 a 179)
}

df_simples = pd.DataFrame(data_simulada_simplificada)

# --- NOVA LÓGICA PARA A VARIÁVEL ALVO (PRIORIZAÇÃO COM CONDIÇÃO DE ESTOQUE) ---
df_simples['Produto_A_Ser_Produzido_Cliente'] = ''

# Lista para armazenar as decisões de produção
decisoes_producao = []

# Primeiro, vamos identificar os pedidos que não podem ser produzidos por falta de estoque
# E os que podem ser produzidos, para aplicar a prioridade neles
pedidos_sem_estoque = df_simples[df_simples['Estoque_OK'] == 'Nao'].copy()
pedidos_com_estoque = df_simples[df_simples['Estoque_OK'] == 'Sim'].copy()

# Para pedidos sem estoque, a decisão é sempre "Aguardar_Estoque"
if not pedidos_sem_estoque.empty:
    pedidos_sem_estoque['Produto_A_Ser_Produzido_Cliente'] = "Aguardar_Estoque"

# Para pedidos com estoque, aplicamos a lógica de prioridade:
if not pedidos_com_estoque.empty:
    # 1. Prioridade para Urgência "Alta"
    pedidos_alta_urgencia = pedidos_com_estoque[pedidos_com_estoque['Urgencia_Pedido'] == 'Alta'].copy()
    
    # 2. Para os demais (Urgência Média/Baixa), prioridade por Dias_Desde_Pedido (mais antigo primeiro)
    pedidos_outras_urgencias = pedidos_com_estoque[pedidos_com_estoque['Urgencia_Pedido'] != 'Alta'].copy()
    pedidos_outras_urgencias = pedidos_outras_urgencias.sort_values(by='Dias_Desde_Pedido', ascending=False)

    # Combinamos os pedidos priorizados para simular a ordem de produção
    # Primeiro os de alta urgência, depois os mais antigos das outras urgências
    pedidos_priorizados_para_producao = pd.concat([pedidos_alta_urgencia, pedidos_outras_urgencias]).reset_index(drop=True)

    # Agora, atribuímos os produtos a serem produzidos para esses pedidos priorizados
    # Vamos simplificar, alternando entre os tipos de etiqueta para os pedidos em ordem de prioridade
    for i in range(len(pedidos_priorizados_para_producao)):
        cliente_atual = pedidos_priorizados_para_producao.loc[i, 'Nome_Cliente']
        produto_escolhido = opcoes_etiquetas[i % len(opcoes_etiquetas)] # Alterna entre os tipos de etiqueta
        pedidos_priorizados_para_producao.loc[i, 'Produto_A_Ser_Produzido_Cliente'] = f"{cliente_atual}_{produto_escolhido}"

    # Recombinar todos os DataFrames: primeiro os que aguardam estoque, depois os que serão produzidos
    df_simples = pd.concat([pedidos_sem_estoque, pedidos_priorizados_para_producao]).sort_index()

# Garantir que a coluna 'Produto_A_Ser_Produzido_Cliente' esteja preenchida para todas as linhas
# Caso alguma condição acima não tenha sido atendida (ex: df_simples vazio), preenche com um valor padrão
if 'Produto_A_Ser_Produzido_Cliente' not in df_simples.columns:
    df_simples['Produto_A_Ser_Produzido_Cliente'] = "Erro_Na_Simulacao" # Para depuração, se algo der errado

# Se houver linhas que não foram preenchidas (o que não deve acontecer com a lógica acima), preencher
df_simples['Produto_A_Ser_Produzido_Cliente'] = df_simples['Produto_A_Ser_Produzido_Cliente'].fillna("Nao_Definido")


print("Cabeçalho dos nossos dados simulados simplificados (as primeiras 5 linhas com a nova priorização):")
print(df_simples.head())

print("\nContagem das decisões combinadas (variável alvo) para ver a distribuição:")
print(df_simples['Produto_A_Ser_Produzido_Cliente'].value_counts())

Cabeçalho dos nossos dados simulados simplificados (as primeiras 5 linhas com a nova priorização):
  Estoque_OK  Dias_Desde_Pedido Urgencia_Pedido Nome_Cliente  \
0        Nao                 16           Baixa    Cliente_4   
0        Sim                 15            Alta    Cliente_1   
1        Nao                 15           Media    Cliente_5   
1        Sim                 15            Alta    Cliente_4   
2        Sim                 22            Alta    Cliente_5   

   Tempo_Medio_Producao_Min Produto_A_Ser_Produzido_Cliente  
0                        93                Aguardar_Estoque  
0                       167        Cliente_1_Etiqueta_105mm  
1                        97                Aguardar_Estoque  
1                       101         Cliente_4_Etiqueta_64mm  
2                       123        Cliente_5_Etiqueta_105mm  

Contagem das decisões combinadas (variável alvo) para ver a distribuição:
Produto_A_Ser_Produzido_Cliente
Aguardar_Estoque            67
Client

In [26]:
# Separando as variáveis (features) da variável alvo (target)
# X_simples serão as variáveis de entrada (o que a IA vai "observar")
# y_simples será a variável que a IA vai "prever" (a decisão combinada)
X_simples = df_simples.drop('Produto_A_Ser_Produzido_Cliente', axis=1) # Remove a coluna alvo de X_simples
y_simples = df_simples['Produto_A_Ser_Produzido_Cliente'] # Seleciona apenas a coluna alvo

# Convertendo variáveis categóricas (de texto) em numéricas usando One-Hot Encoding
# Isso é necessário porque algoritmos de ML trabalham melhor com números
X_simples = pd.get_dummies(X_simples, columns=[
    'Estoque_OK',
    'Urgencia_Pedido',
    'Nome_Cliente'
])

print("Cabeçalho dos dados preparados (X_simples) - agora com colunas numéricas:")
print(X_simples.head())

print("\nInformações sobre os tipos de dados de X_simples:")
print(X_simples.info())

Cabeçalho dos dados preparados (X_simples) - agora com colunas numéricas:
   Dias_Desde_Pedido  Tempo_Medio_Producao_Min  Estoque_OK_Nao  \
0                 16                        93            True   
0                 15                       167           False   
1                 15                        97            True   
1                 15                       101           False   
2                 22                       123           False   

   Estoque_OK_Sim  Urgencia_Pedido_Alta  Urgencia_Pedido_Baixa  \
0           False                 False                   True   
0            True                  True                  False   
1           False                 False                  False   
1            True                  True                  False   
2            True                  True                  False   

   Urgencia_Pedido_Media  Nome_Cliente_Cliente_1  Nome_Cliente_Cliente_2  \
0                  False                   False        

In [27]:
from sklearn.model_selection import train_test_split

# Dividindo os dados em conjuntos de treinamento e teste
# test_size=0.20 significa que 20% dos dados serão para teste e 80% para treinamento
# random_state=42 garante que a divisão seja a mesma toda vez que você rodar o código (para reprodutibilidade)
X_train, X_test, y_train, y_test = train_test_split(X_simples, y_simples, test_size=0.20, random_state=42)

print(f"Tamanho total dos dados de entrada (X_simples): {X_simples.shape}")
print(f"Tamanho dos dados de treinamento (X_train): {X_train.shape}")
print(f"Tamanho dos dados de teste (X_test): {X_test.shape}")
print(f"Tamanho dos dados alvo de treinamento (y_train): {y_train.shape}")
print(f"Tamanho dos dados alvo de teste (y_test): {y_test.shape}")

Tamanho total dos dados de entrada (X_simples): (200, 12)
Tamanho dos dados de treinamento (X_train): (160, 12)
Tamanho dos dados de teste (X_test): (40, 12)
Tamanho dos dados alvo de treinamento (y_train): (160,)
Tamanho dos dados alvo de teste (y_test): (40,)


In [28]:
from sklearn.tree import DecisionTreeClassifier

# 1. Criar o modelo da Árvore de Decisão
# Usamos random_state=42 para garantir que o treinamento seja o mesmo toda vez que você rodar o código.
model = DecisionTreeClassifier(random_state=42)

# 2. Treinar o modelo
# A função .fit() é onde o modelo aprende. Ele usa X_train (as características)
# para aprender a prever y_train (as decisões de produção).
model.fit(X_train, y_train)

print("Modelo de Árvore de Decisão treinado com sucesso!")

Modelo de Árvore de Decisão treinado com sucesso!


In [29]:
# 1. Fazer previsões nos dados de teste
# O modelo agora tenta prever a decisão para os dados que ele nunca viu
y_pred = model.predict(X_test)

# 2. Avaliar a acurácia do modelo
# Comparamos as previsões do modelo (y_pred) com as respostas corretas reais (y_test)
acuracia = accuracy_score(y_test, y_pred)

print(f"A acurácia do modelo nos dados de teste é: {acuracia:.2f}")

A acurácia do modelo nos dados de teste é: 0.60
